# 08 - Evaluation: Retrieval, Generation, and Judge

## Definition
RAG evaluation measures retrieval quality and generation quality as separate but linked objectives.

## Theory
Poor retrieval creates an upper bound on generation quality regardless of model fluency.

## Motivation
Without robust metrics, improvements may be cosmetic rather than factual.

## Architecture
Query set -> retrieval metrics -> generation metrics -> LLM judge -> hallucination analysis.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [ ]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [ ]:
bundle = runner.ingest_dataset()
chunking = runner.run_chunking(bundle['documents'][:320], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

queries = bundle['queries'][:24]
summary, frames = runner.evaluator.run_full_evaluation(
    queries=queries,
    top_k=6,
    retrieval_limit=18,
    generation_limit=6,
    judge_limit=6,
)

{
    'retrieval': summary.retrieval,
    'generation': summary.generation,
    'judge': summary.judge,
}


In [ ]:
hall = runner.evaluator.evaluate_hallucination_reduction(queries=queries[:8], max_queries=8)
hall[['groundedness_delta', 'rag_faithfulness', 'no_rag_faithfulness']].describe()
